In [ ]:
import matplotlib.colors as mplcol
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
from scipy.io import loadmat
import numpy as np
import sys

sys.path.append("..")
from mienc import Corrector

In [ ]:
filename = "/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat"
mat = loadmat(filename)["subj_tcs"]
step_num, reg_num, subj_num = mat.shape

name = filename.split("_")[1]
reg_pair = int(reg_num * (reg_num - 1) / 2)

num_mats = 20
block_size = int(step_num / num_mats)
corr1 = np.empty((num_mats, reg_pair, subj_num))
for i in np.arange(num_mats):
    for j in range(subj_num):
        corr1[i, :, j] = np.corrcoef(
            mat[block_size * i : block_size * (i + 1), :, j], rowvar=False
        )[np.triu_indices(reg_num, 1)]

corr2 = np.zeros((num_mats, num_mats, subj_num))
for i in range(subj_num):
    tmp = np.corrcoef(corr1[:, :, i], rowvar=True)
    np.fill_diagonal(tmp, np.nan)
    corr2[:, :, i] = tmp[:, :]

In [ ]:
fig, ax = plt.subplots(10, 10, figsize=(18, 20))
for i, subj in enumerate(sorted(np.random.choice(245, 100, False))):
    plt.sca(ax[i // 10, i % 10])
    plt.imshow(corr2[:, :, subj])
    # plt.colorbar(shrink=0.8)
    plt.xticks(np.array([5, 10, 15, 20]) - 1)
    plt.yticks([])
    plt.title(subj)
plt.suptitle(name)
print(name)
plt.subplots_adjust(wspace=0, hspace=0.1)
plt.show()

In [ ]:
from tqdm import tqdm
import community
import networkx

fig, ax = plt.subplots(10, 10, figsize=(18, 20))
for i, subj in tqdm(enumerate(sorted(np.random.choice(245, 100, False))), total=100):
    brova = corr2[:, :, subj]
    corr_plt = (brova[:, :] + brova[:, :].T) / 2  # made symmetric
    np.fill_diagonal(corr_plt, 1)
    # nc = []
    # fro = []
    # all_labels = []
    # for t in np.linspace(0.2, 0.95, 16, True):
    #     dissimilarity = 1 - np.abs(corr_plt)
    #     hierarchy = linkage(squareform(dissimilarity), method='average')
    #     labels = fcluster(hierarchy, t, criterion='distance')
    #     all_labels.append(labels)
    #     ideal = np.array([labels == c for c in labels])
    #     nc.append(len(np.unique(labels)))
    #     fro.append(np.linalg.norm(corr_plt-ideal, ord="fro"))
    # pts = np.array([nc, fro])
    # d0 = pts[:,:1]-pts
    # d1 =pts[:,-1:]-pts
    # elbow = np.nanargmax([(d1[:,c]@d0[:,c])/(np.linalg.norm(d1[:,c])*np.linalg.norm(d0[:,c])) for c in range(len(nc))])
    # order = np.concatenate([np.where(all_labels[elbow]==c)[0] for c in np.unique(all_labels[elbow])])
    grap = networkx.Graph(np.abs(corr_plt))
    p = community.community_louvain.best_partition(grap)
    labels = np.array([p[n] for n in p])
    order = np.concatenate([np.where(labels == c)[0] for c in np.unique(labels)])
    plt.sca(ax[i // 10, i % 10])
    np.fill_diagonal(corr_plt, 0)
    plt.imshow(corr_plt[:, order][order, :])
    # plt.colorbar(shrink=0.8)
    plt.xticks(np.array([5, 10, 15, 20]) - 1)
    plt.yticks([])
    plt.title(f"{subj} - {np.unique(labels).size}")
plt.suptitle(name)
print(name)
plt.subplots_adjust(wspace=0, hspace=0.1)
plt.show()

In [ ]:
print(order)
plt.imshow(corr_plt)
# plt.colorbar(shrink=0.8)
plt.xticks(np.array([5, 10, 15, 20]) - 1)
plt.yticks([])
plt.title(subj)
plt.suptitle(name)
print(name)
plt.subplots_adjust(wspace=0, hspace=0.1)
plt.show()

In [ ]:
TL = 219
int_corr = corr2[:, :, TL]

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plt.sca(ax)
plt.imshow(int_corr[:, :])
plt.colorbar(shrink=0.8)
plt.title(TL)
plt.xticks(np.array([1, 5, 10, 15, 20]) - 1)
plt.yticks([])
plt.suptitle(name)
plt.subplots_adjust(wspace=0, hspace=0)
plt.show()

corr_plt = (int_corr[:, :] + int_corr[:, :].T) / 2  # made symmetric
np.fill_diagonal(corr_plt, 1)  # put 1 on the diagonal
dissimilarity = 1 - np.abs(corr_plt)
hierarchy = linkage(squareform(dissimilarity), method="average")
labels = fcluster(hierarchy, 0.8, criterion="distance")

n_class = len(set(labels))
new_lab = {}
top = 0
for l in labels:
    if l not in new_lab:
        new_lab[l] = top
        top += 1
        if top == n_class:
            break
labels[:] = [new_lab[i] for i in labels]
# labels[-1]=5
colors = np.repeat(labels, 15) - 1

# define the bins and normalize
bounds = np.linspace(0, n_class, n_class + 1)
norm = mplcol.BoundaryNorm(bounds, n_class)  # cmap.N)#plt.cm.Set3.N


plt.figure(figsize=(4, 1))
plt.imshow(labels[np.newaxis, :] - 1, cmap=plt.cm.Set3, norm=norm)
plt.yticks([])
plt.show()

In [ ]:
plt.imshow()

In [ ]:
# np.fill_diagonal(corr_plt, np.nan) 
        if nc[-1]==1:
            fro.append(-np.nanmean(corr_plt))
        elif nc[-1]==20:
            fro.append(1-np.nanmean(corr_plt))
        else:
            fro.append(np.nanmean(corr_plt[ideal])-np.nanmean(corr_plt[1-ideal]))#

In [ ]:
for TL in range(10):
    brova = corr2[:, :, TL]

    corr_plt = (brova[:, :] + brova[:, :].T) / 2  # made symmetric
    nc = []
    fro = []
    all_labels = []
    for t in np.linspace(0.2, 0.95, 16, True):
        np.fill_diagonal(corr_plt, 1)
        dissimilarity = 1 - np.abs(corr_plt)
        hierarchy = linkage(squareform(dissimilarity), method="average")
        labels = fcluster(hierarchy, t, criterion="distance")
        all_labels.append(labels)
        ideal = np.array([labels == i for i in labels])
        nc.append(len(np.unique(labels)))
        fro.append(np.linalg.norm(corr_plt - ideal, ord="fro"))
    pts = np.array([nc, fro])
    d0 = pts[:, :1] - pts
    d1 = pts[:, -1:] - pts
    elbow = np.nanargmax(
        [
            (d1[:, i] @ d0[:, i])
            / (np.linalg.norm(d1[:, i]) * np.linalg.norm(d0[:, i]))
            for i in range(len(nc))
        ]
    )
    print(all_labels[elbow])
    plt.scatter(nc, fro)
    plt.scatter(nc[elbow], fro[elbow], marker="*", color="r")
    plt.title(TL)
    plt.show()

In [ ]:
order = np.concatenate([np.where(labels == i)[0] for i in np.unique(labels)])
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
plt.sca(ax[0])
plt.imshow(int_corr[order, :][:, order])
# plt.colorbar(shrink=0.8)
plt.title(TL)
plt.sca(ax[1])
plt.imshow(int_corr)
# plt.colorbar(shrink=0.8)
plt.title(TL)
plt.xticks(np.array([1, 5, 10, 15, 20]) - 1)
plt.yticks([])
plt.suptitle(name)
plt.subplots_adjust(wspace=0, hspace=0)
plt.show()

In [ ]:
res = np.load(
    f"../../NonLinearityResults/eso245_aal_strin_90_bin9/patient{TL:02}_9.npy"
)
correct = Corrector(
    9,
    400,
    folder_name="../../NonLinearityResults/eso245_aal_strin_90_bin9/",
    cache_dir="../cache",
    config="config.ini",
)
correct.compute_correction()
res_cor = correct.correct(res)

In [ ]:
TMI = res_cor[:, 0]
GMI = np.mean(res_cor[:, 1:], axis=1)
RNL = 1 - GMI / TMI
plt.scatter(GMI, TMI)
RNL[GMI - TMI < 0.1] = 0
topp = np.argsort(RNL)[-6:]

In [ ]:
corr_plt = (int_corr[:, :] + int_corr[:, :].T) / 2  # made symmetric
np.fill_diagonal(corr_plt, 1)  # put 1 on the diagonal
dissimilarity = 1 - np.abs(corr_plt)
hierarchy = linkage(squareform(dissimilarity), method="average")
labels = fcluster(hierarchy, 0.8, criterion="distance")
colors = np.repeat(labels, 20) - 1

plt.figure(figsize=(4, 1))
plt.imshow(labels[np.newaxis, :] - 1, cmap=plt.cm.Set3, norm=norm)
plt.yticks([])
plt.show()
for r1, r2 in np.array(np.triu_indices(reg_num, 1))[:, topp].T:
    plt.scatter(
        mat[:, r1, TL],
        mat[:, r2, TL],
        c=colors,
        cmap=plt.cm.Set3,
        norm=norm,
        alpha=0.8,
        s=15,
    )  # plt.cm.Set3, alpha=0.7)
    plt.colorbar()
    plt.title(f"{r1} - {r2}")
    plt.show()